In [1]:
import os
import re
import pandas as pd
from tqdm import tqdm
import numpy as np
import psycopg2
import matplotlib.pyplot as plt
import seaborn as sns
import nest_asyncio

In [2]:
icustays = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/icu/icustays.csv.gz')
admissions = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/hosp/admissions.csv.gz')
pts = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/hosp/patients.csv.gz')

In [3]:
pts['yob']= pts['anchor_year'] - pts['anchor_age']  # get yob to ensure a given visit is from an adult
pts['min_valid_year'] = pts['anchor_year'] + (2022 - pts['anchor_year_group'].str.slice(start=-4).astype(int))

In [4]:
pts = pd.merge(icustays,pts,on='subject_id',how='left')
pts = pd.merge(pts,admissions[['hadm_id', 'admittime', 'dischtime', 'deathtime',
       'admission_type', 'admit_provider_id', 'admission_location',
       'discharge_location', 'insurance', 'language', 'marital_status', 'race',
       'edregtime', 'edouttime', 'hospital_expire_flag']],on='hadm_id',how='left')
pts.shape

(94458, 29)

In [5]:
pts['admittime'] = pd.to_datetime(pts['admittime'])
pts['Age']=pts['admittime'].dt.year - pts['yob']
pts = pts.loc[pts['Age'] >= 18]
pts.shape

(94458, 30)

In [6]:
pts = pts[pts.los>=1]
pts.shape

(74829, 30)

In [12]:
df = pts.copy()
print(len(df.subject_id.unique()),len(df.hadm_id.unique()),len(df.stay_id.unique()))

54551 68546 74829


In [13]:
microbiologyevents = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/hosp/microbiologyevents.csv.gz')
microevents = microbiologyevents[microbiologyevents['hadm_id'].isin(df['hadm_id'])]
del(microbiologyevents)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_42516\3507416237.py:1: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  microbiologyevents = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/hosp/microbiologyevents.csv.gz')


In [ ]:
microevents = microevents[['microevent_id','micro_specimen_id', 'subject_id', 'hadm_id',
                           'chartdate', 'charttime', 'spec_itemid',
       'spec_type_desc', 'storedate', 'storetime', 
       'test_name', 'org_itemid', 'org_name','ab_name', 
       'interpretation']]
microevents.shape

(776142, 15)

In [15]:
merged = pd.merge(microevents, df[['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime',
                                   'admittime','los']], on =['subject_id', 'hadm_id'], how='left')

merged['intime'] = pd.to_datetime(merged['intime'])
merged['outtime'] = pd.to_datetime(merged['outtime'])
merged['admittime'] = pd.to_datetime(merged['admittime'])

merged['charttime'] = pd.to_datetime(merged['charttime'])
merged['storetime'] = pd.to_datetime(merged['storetime'])

# SSC during ICU

In [ ]:
SSC_icu_window_criteria = merged['charttime'].between(
    merged['intime']- pd.DateOffset(1),
    merged['outtime'])

merged = merged[SSC_icu_window_criteria]

In [19]:
merged = merged.sort_values(['stay_id','intime','charttime','org_name']).drop_duplicates(['stay_id','charttime','test_name','org_name','ab_name', 'interpretation']) 

In [ ]:
merged.to_csv('/MIMIC_IV/1_SSC_ICU_IV.csv',index=False)

In [22]:
print(merged.shape,len(merged.stay_id.unique()),len(merged.microevent_id.unique()))

(531643, 20) 56899 530542


### BI

In [23]:
BI = merged.dropna(subset = ['org_name'])
print(BI.shape,len(pd.unique(BI.subject_id)),len(pd.unique(BI.hadm_id)))

(198437, 20) 16396 18839


In [24]:
org_name_sel = BI[['org_itemid', 'org_name']]
org_name_sel = org_name_sel.drop_duplicates(subset=['org_name'], keep='first')
org_name_sel.shape

(379, 2)

In [ ]:
def classify_bacteria(org_name):
    org_name = org_name.upper()  
    
    if "STAPH AUREUS" in org_name:
        return "Staphylococcus aureus"
    elif any(keyword in org_name for keyword in [
        "ENTEROBACTER", "ESCHERICHIA", "KLEBSIELLA", 
        "CITROBACTER", "MORGANELLA", "PROVIDENCIA", 
        "SALMONELLA"]):
        return "Enterobacteriaceae"
    elif "ENTEROCOCCUS" in org_name:
        return "Enterococcus spp."
    elif "PSEUDOMONAS AERUGINOSA" in org_name:
        return "Pseudomonas aeruginosa"
    elif "ACINETOBACTER" in org_name:
        return "Acinetobacter spp."
    else:
        return "Others"

In [26]:
org_name_sel['org_category'] = org_name_sel['org_name'].apply(classify_bacteria)

In [27]:
org_name_sel[org_name_sel.org_category=='Staphylococcus aureus']

,org_itemid,org_name,org_category
434675,80023.0,STAPH AUREUS COAG +,Staphylococcus aureus
828297,80293.0,POSITIVE FOR METHICILLIN RESISTANT STAPH AUREUS,Staphylococcus aureus


In [28]:
org_name_sel[org_name_sel.org_category=='Acinetobacter spp.']

,org_itemid,org_name,org_category
110359,80304.0,ACINETOBACTER BAUMANNII COMPLEX,Acinetobacter spp.
623069,80216.0,ACINETOBACTER SP.,Acinetobacter spp.
229676,80194.0,ACINETOBACTER BAUMANNII,Acinetobacter spp.
719058,90813.0,ACINETOBACTER RADIORESISTENS,Acinetobacter spp.
88542,90287.0,ACINETOBACTER LWOFFII,Acinetobacter spp.
715348,91024.0,ACINETOBACTER URSINGII,Acinetobacter spp.


In [30]:
MDR_CAL_cha = BI[['stay_id','micro_specimen_id','org_name','ab_name','interpretation']].set_index(['stay_id','micro_specimen_id','org_name','ab_name'], append=True).unstack(-1).reset_index()
MDR_CAL_cha = MDR_CAL_cha.iloc[:,1:]
MDR_CAL_cha.columns = ['stay_id','micro_specimen_id','org_name'] + MDR_CAL_cha['interpretation'].columns.to_list()
MDR_CAL_cha.drop(columns = np.nan, inplace = True)
MDR_CAL_cha.shape

(198437, 50)

In [ ]:
MDR_CAL_cha_test = MDR_CAL_cha.copy()
ab_list = MDR_CAL_cha.columns.to_list()
ab_list[0:3]

['stay_id', 'micro_specimen_id', 'org_name']

In [32]:
del ab_list[0:3]
for i in ab_list:
    MDR_CAL_cha_test[i] = MDR_CAL_cha_test[i].astype(str)

In [33]:
MDR_CAL_cha_test = MDR_CAL_cha_test.groupby(['stay_id','micro_specimen_id','org_name']).sum().reset_index()

In [34]:
def find_r(x):
    if re.search('R',x):
        return 'R'
    elif re.search('I',x):
        return 'I'
    elif re.search('S',x):
        return 'S'
    else:
        return np.nan

for i in ab_list:
    MDR_CAL_cha_test[i] = MDR_CAL_cha_test[i].apply(lambda x:find_r(x))

MDR_CAL_conc = MDR_CAL_cha_test.drop_duplicates()
MDR_CAL_conc.shape

(55110, 50)

In [35]:
MDR_CAL_cat = pd.merge(MDR_CAL_conc,org_name_sel[['org_name','org_category']],\
how = 'left', on = ['org_name'])#.org_category.value_counts()

In [36]:
MDR_CAL_cat.org_category.value_counts()

Others                    31793
Staphylococcus aureus      9488
Enterobacteriaceae         7202
Enterococcus spp.          3215
Pseudomonas aeruginosa     2989
Acinetobacter spp.          423
Name: org_category, dtype: int64

In [ ]:
org_pieces = dict(list(org_name_sel['org_name'].groupby(org_name_sel['org_category'])))

St_name = org_pieces['Staphylococcus aureus'].tolist()
Eb_name = org_pieces['Enterobacteriaceae'].tolist()
Ec_name = org_pieces['Enterococcus spp.'].tolist()
Ps_name = org_pieces['Pseudomonas aeruginosa'].tolist()
Ac_name = org_pieces['Acinetobacter spp.'].tolist()

Eb_exc_amino_list = ['PROVIDENCIA RETTGERI', 'PROVIDENCIA STUARTII']
Eb_exc_cefazolin_list = [
    'CITROBACTER FREUNDII COMPLEX', 'ENTEROBACTER AEROGENES',
    'ENTEROBACTER CLOACAE COMPLEX', 'ENTEROBACTER CLOACAE', 'HAFNIA ALVEI',
    'MORGANELLA MORGANII', 'PROTEUS PENNERI', 'PROTEUS VULGARIS',
    'PROTEUS VULGARIS GROUP', 'PROVIDENCIA RETTGERI', 'PROVIDENCIA STUARTII',
    'SERRATIA MARCESCENS'
]
Eb_exc_cefuroxime_list = [
    'MORGANELLA MORGANII', 'PROTEUS PENNERI', 'PROTEUS VULGARIS',
    'PROTEUS VULGARIS GROUP', 'SERRATIA MARCESCENS'
]
Eb_exc_ampicillin_list = [
    'CITROBACTER KOSERI', 'CITROBACTER FREUNDII COMPLEX',
    'ENTEROBACTER AEROGENES', 'ENTEROBACTER CLOACAE COMPLEX',
    'ENTEROBACTER CLOACAE', 'HAFNIA ALVEI', 'KLEBSIELLA PNEUMONIAE',
    'KLEBSIELLA OXYTOCA', 'MORGANELLA MORGANII', 'PROTEUS PENNERI',
    'PROTEUS VULGARIS', 'PROTEUS VULGARIS GROUP', 'PROVIDENCIA RETTGERI',
    'PROVIDENCIA STUARTII', 'SERRATIA MARCESCENS'
]
Eb_exc_ampicillinsulbactam_list = [
    'CITROBACTER FREUNDII COMPLEX', 'CITROBACTER KOSERI',
    'ENTEROBACTER AEROGENES', 'ENTEROBACTER CLOACAE COMPLEX',
    'ENTEROBACTER CLOACAE', 'HAFNIA ALVEI', 'PROVIDENCIA RETTGERI',
    'SERRATIA MARCESCENS'
]
Eb_exc_teracycline_list = [
    'MORGANELLA MORGANII', 'PROTEUS MIRABILIS','PROTEUS PENNERI',
    'PROTEUS VULGARIS', 'PROTEUS VULGARIS GROUP', 'PROVIDENCIA RETTGERI',
    'PROVIDENCIA STUARTII'
]

In [ ]:
def MDR(df):
    if df['org_name'] in St_name:
        if df['org_name'] == 'POSITIVE FOR METHICILLIN RESISTANT STAPH AUREUS':
            return 'MRSA', 1
        elif df['org_name'] == 'S. AUREUS POSITIVE; MRSA POSITIVE':
            return 'MRSA', 1
        else:
            St_dict = {}
            St_sum = 0
            St_ab_list = []

            if df['GENTAMICIN'] == 'I' or df['GENTAMICIN'] == 'R':
                St_dict['Aminoglycosides'] = 1

            if df['RIFAMPIN'] == 'I' or df['RIFAMPIN'] == 'R':
                St_dict['Ansamycins'] = 1
    
            if (df[['AMPICILLIN','AMPICILLIN/SULBACTAM','CEFAZOLIN',\
                'CEFEPIME','CEFTAZIDIME','CEFTRIAXONE','CEFUROXIME',\
                'OXACILLIN','PENICILLIN G','PIPERACILLIN','PIPERACILLIN/TAZO']] == 'I').any() or \
                (df[['AMPICILLIN','AMPICILLIN/SULBACTAM','CEFAZOLIN',\
                'CEFEPIME','CEFTAZIDIME','CEFTRIAXONE','CEFUROXIME',\
                'OXACILLIN','PENICILLIN G','PIPERACILLIN','PIPERACILLIN/TAZO']] == 'R').any():
                St_dict['LactamsCephamycins'] = 1
        
            if df['CIPROFLOXACIN']=='I' or df['CIPROFLOXACIN']=='R':
                St_dict['Fluoroquinolones'] = 1
    
            if df['TRIMETHOPRIM/SULFA']=='I' or df['TRIMETHOPRIM/SULFA']=='R':
                St_dict['Folate'] = 1
      
            if df['VANCOMYCIN'] == 'I' or df['VANCOMYCIN'] == 'R':
                St_dict['Glycopeptides'] = 1
        
            if df['CLINDAMYCIN'] == 'I' or df['CLINDAMYCIN'] == 'R':
                St_dict['Lincosamides'] = 1
    
            if df['DAPTOMYCIN'] == 'I' or df['DAPTOMYCIN'] == 'R':
                St_dict['Lipopeptides'] = 1
 
            if df['ERYTHROMYCIN'] == 'I' or df['ERYTHROMYCIN'] == 'R':
                St_dict['Macrolides'] = 1
        
            if df['LINEZOLID'] == 'I' or df['LINEZOLID'] == 'R':
                St_dict['Oxazolidinones'] = 1
    
            if df['TETRACYCLINE'] == 'I' or df['TETRACYCLINE'] == 'R':
                St_dict['Tetracyclines'] = 1
            
            for i in St_dict:
                St_sum += St_dict[i]
                St_ab_list.append(i)
            if St_sum>=3:
                return ";".join(St_ab_list), 1
            else:
                return ";".join(St_ab_list), 0
  
    elif df['org_name'] in Ec_name:
        Ec_dict = {}
        Ec_sum = 0
        Ec_ab_list = []
  
        if (df[['IMIPENEM','MEROPENEM']] == 'I').any() or \
                (df[['IMIPENEM','MEROPENEM']] == 'R').any():
            if df['org_name'] != 'ENTEROCOCCUS FAECIUM': 

                Ec_dict['Carbapenems'] = 1

        if (df[['CIPROFLOXACIN','LEVOFLOXACIN']]=='I').any() or \
                (df[['CIPROFLOXACIN','LEVOFLOXACIN']]=='R').any():
            Ec_dict['Fluroquinolones'] = 1
 
        if df['VANCOMYCIN'] == 'I' or df['VANCOMYCIN'] == 'R':
            Ec_dict['Glycopeptides'] = 1

        if df['DAPTOMYCIN'] == 'I' or df['DAPTOMYCIN'] == 'R':
            Ec_dict['Lipopeptides'] = 1
    
        if df['LINEZOLID'] == 'I' or df['LINEZOLID'] == 'R':
            Ec_dict['Oxazolidinones'] = 1

        if df['AMPICILLIN'] == 'I' or df['AMPICILLIN'] == 'R':
            Ec_dict['Penicillins'] = 1

        if df['TETRACYCLINE'] == 'I' or df['TETRACYCLINE'] == 'R':
            Ec_dict['Tetracyclines'] = 1
        
        for i in Ec_dict:
            Ec_sum += Ec_dict[i]
            Ec_ab_list.append(i)
        if Ec_sum>=3:
            return ";".join(Ec_ab_list), 1
        else:
            return ";".join(Ec_ab_list), 0

    elif df['org_name'] in Eb_name:
        Eb_dict = {}
        Eb_sum = 0
        Eb_ab_list = []
   
        if (df[['GENTAMICIN','TOBRAMYCIN']] == 'I').any() \
            or (df[['GENTAMICIN','TOBRAMYCIN']] == 'R').any():
            if df['org_name'] not in Eb_exc_amino_list:
                Eb_dict['Aminoglycosides'] = 1
        if df['AMIKACIN'] == 'I' or df['AMIKACIN'] == 'R':
            Eb_dict['Aminoglycosides'] = 1

        if df['PIPERACILLIN/TAZO'] == 'I' or df['PIPERACILLIN/TAZO'] == 'R':
            Eb_dict['AntiPenicillins&Lactamaseinhibitors'] = 1

        if (df[['IMIPENEM','MEROPENEM']] == 'I').any() or \
            (df[['IMIPENEM','MEROPENEM']] == 'R').any():
            Eb_dict['Carbapenems'] = 1
  
        if df['CEFAZOLIN']=='I' or df['CEFAZOLIN']=='R':
            if df['org_name'] not in Eb_exc_cefazolin_list:
                Eb_dict['NESCephalosporins'] = 1
        if df['CEFUROXIME']=='I' or df['CEFUROXIME']=='R':
            if df['org_name'] not in Eb_exc_cefuroxime_list:
                Eb_dict['NESCephalosporins'] = 1
   
        if (df[['CEFTRIAXONE','CEFTAZIDIME','CEFEPIME']]=='I').any() \
            or (df[['CEFTRIAXONE','CEFTAZIDIME','CEFEPIME']]=='R').any():
            Eb_dict['ESCephalosporins'] = 1
     
        if df['CIPROFLOXACIN'] == 'I' or df['CIPROFLOXACIN'] == 'R':
            Eb_dict['Fluoroquinolones'] = 1

        if df['TRIMETHOPRIM/SULFA']=='I' or df['TRIMETHOPRIM/SULFA']=='R':
            Eb_dict['Folate'] = 1

        if df['AMPICILLIN']=='I' or df['AMPICILLIN']=='R':
            if df['org_name'] not in Eb_exc_ampicillin_list:
                Eb_dict['Penicillins'] = 1 
   
        if df['AMPICILLIN/SULBACTAM']=='I' or df['AMPICILLIN/SULBACTAM']=='R':
            if df['org_name'] not in Eb_exc_ampicillinsulbactam_list:
                Eb_dict['Penicillins&Lactamaseinhibitors'] = 1

        if df['VANCOMYCIN'] == 'I' or df['VANCOMYCIN'] == 'R':
            Eb_dict['Glycopeptides'] = 1

        if df['TETRACYCLINE'] == 'I' or df['TETRACYCLINE'] == 'R':
            if df['org_name'] not in Eb_exc_teracycline_list:
                Eb_dict['Tetracyclines'] = 1
        
        for i in Eb_dict:
            Eb_sum += Eb_dict[i]
            Eb_ab_list.append(i)
        if Eb_sum>=3:
            return ";".join(Eb_ab_list), 1
        else:
            return ";".join(Eb_ab_list), 0

    elif df['org_name'] in Ps_name:
        Ps_dict = {}
        Ps_sum = 0
        Ps_ab_list = []

        if (df[['GENTAMICIN','TOBRAMYCIN','AMIKACIN']] == 'I').any() \
            or (df[['GENTAMICIN','TOBRAMYCIN','AMIKACIN']] == 'R').any():
            Ps_dict['Aminoglycosides'] = 1    

        if (df[['IMIPENEM','MEROPENEM']] == 'I').any() or \
                (df[['IMIPENEM','MEROPENEM']] == 'R').any():
            Ps_dict['Carbapenems'] = 1
     
        if (df[['CEFTAZIDIME','CEFEPIME']] == 'I').any() or \
                (df[['CEFTAZIDIME','CEFEPIME']] == 'R').any():
            Ps_dict['Cephalosporins'] = 1      
       
        if (df[['CIPROFLOXACIN','LEVOFLOXACIN']]=='I').any() or \
                (df[['CIPROFLOXACIN','LEVOFLOXACIN']]=='R').any():
            Ps_dict['Fluroquinolones'] = 1
     
        if df['PIPERACILLIN/TAZO'] == 'I' or df['PIPERACILLIN/TAZO'] == 'R':
            Ps_dict['Penicillins&lactamaseinhibitors'] = 1
      
        for i in Ps_dict:
            Ps_sum += Ps_dict[i]
            Ps_ab_list.append(i)
        if Ps_sum>=3:
            return ";".join(Ps_ab_list), 1
        else:
            return ";".join(Ps_ab_list), 0

    elif df['org_name'] in Ac_name:
        Ac_dict = {}
        Ac_sum = 0
        Ac_ab_list = []
      
        if (df[['GENTAMICIN','TOBRAMYCIN','AMIKACIN']] == 'I').any() \
            or (df[['GENTAMICIN','TOBRAMYCIN','AMIKACIN']] == 'R').any():
            Ac_dict['Aminoglycosides'] = 1    
   
        if (df[['IMIPENEM','MEROPENEM']] == 'I').any() or \
                (df[['IMIPENEM','MEROPENEM']] == 'R').any():
            Ac_dict['Carbapenems'] = 1
     
        if (df[['CIPROFLOXACIN','LEVOFLOXACIN']]=='I').any() or \
                (df[['CIPROFLOXACIN','LEVOFLOXACIN']]=='R').any():
            Ac_dict['Fluroquinolones'] = 1
    
        if df['PIPERACILLIN/TAZO'] == 'I' or df['PIPERACILLIN/TAZO'] == 'R':
            Ac_dict['Penicillins&lactamaseinhibitors'] = 1
   
        if (df[['CEFTAZIDIME','CEFEPIME']] == 'I').any() or \
                (df[['CEFTAZIDIME','CEFEPIME']] == 'R').any():
            Ac_dict['ESCephalosporins'] = 1
     
        if df['TRIMETHOPRIM/SULFA']=='I' or df['TRIMETHOPRIM/SULFA']=='R':
            Ac_dict['Folate'] = 1
     
        if df['AMPICILLIN/SULBACTAM']=='I' or df['AMPICILLIN/SULBACTAM']=='R':
            Ac_dict['Penicillins&lactamaseinhibitors'] = 1        
  
        if df['TETRACYCLINE'] == 'I' or df['TETRACYCLINE'] == 'R':
            Ac_dict['Tetracyclines'] = 1        
        for i in Ac_dict:
            Ac_sum += Ac_dict[i]
            Ac_ab_list.append(i)
        if Ac_sum>=3:
            return ";".join(Ac_ab_list), 1
        else:
            return ";".join(Ac_ab_list), 0
    else:
        return np.nan, np.nan

In [39]:
MDR_CAL_cat[['resistant_ab_cat','MDR']] = MDR_CAL_cat.apply(MDR,axis = 1,result_type="expand")

In [ ]:
MDR_CAL_cat['MDR'] = MDR_CAL_cat[['org_name','VANCOMYCIN','MDR']].\
apply(lambda x: 1 if ((x[0].find('ENTEROCOCCUS'))!=-1) \
and ((x[1] =='R') or (x[1] =='I')) \
else x[2],axis = 1)

MDR_CAL_cat = MDR_CAL_cat.drop_duplicates()

for i,j in zip(['Staphylococcus aureus', 'Enterobacteriaceae', 'Enterococcus spp.', 
                'Pseudomonas aeruginosa','Acinetobacter spp.'],\
        [St_name, Eb_name, Ec_name, Ps_name, Ac_name]):
    print('-----------%s------------'%i)
    print(MDR_CAL_cat[MDR_CAL_cat.org_name.isin(j)].MDR.value_counts())

-----------Staphylococcus aureus------------
0.0    5852
1.0    3636
Name: MDR, dtype: int64
-----------Enterobacteriaceae------------
0.0    5333
1.0    1869
Name: MDR, dtype: int64
-----------Enterococcus spp.------------
0.0    2140
1.0    1075
Name: MDR, dtype: int64
-----------Pseudomonas aeruginosa------------
0.0    2263
1.0     726
Name: MDR, dtype: int64
-----------Acinetobacter spp.------------
0.0    280
1.0    143
Name: MDR, dtype: int64


In [41]:
len(MDR_CAL_cat.stay_id.unique())

20032

In [ ]:
MDR_CAL_cat.to_csv('/MIMIC_IV/1_MDR_ICU_IV.csv',index=False)

In [43]:
MDR_label = MDR_CAL_cat[['stay_id','micro_specimen_id', 'org_name','org_category', 'resistant_ab_cat', 'MDR']]
print(len(MDR_label.stay_id.unique()),len(MDR_label.micro_specimen_id.unique()))

20032 43776


In [44]:
MDR_label[MDR_label.org_category == 'Others'].MDR.value_counts()

Series([], Name: MDR, dtype: int64)

In [45]:
MDR_label = MDR_label.dropna(subset=['MDR'])
MDR_label.shape

(23317, 6)

In [47]:
MDR_stay_id = MDR_label[['stay_id','org_category','MDR']]
MDR_stay_id = MDR_stay_id.sort_values(['stay_id','org_category','MDR'],ascending = False)
MDR_stay_id.org_category.value_counts()

Staphylococcus aureus     9488
Enterobacteriaceae        7202
Enterococcus spp.         3215
Pseudomonas aeruginosa    2989
Acinetobacter spp.         423
Name: org_category, dtype: int64

In [48]:
MDR_stay_id = MDR_stay_id.drop_duplicates(subset = ['stay_id','org_category'],keep = 'first')
MDR_stay_id.shape

(13851, 3)

In [49]:
MDR_stay_id=MDR_stay_id.set_index(['stay_id', 'org_category'])['MDR'].unstack().reset_index()
MDR_stay_id['MDR_label'] = MDR_stay_id[
    ['Acinetobacter spp.', 'Enterobacteriaceae', 'Enterococcus spp.', 
     'Pseudomonas aeruginosa', 'Staphylococcus aureus']].eq(1).any(axis=1).astype(int)
MDR_stay_id.shape

(11608, 7)

In [50]:
MDR_stay_id.MDR_label.value_counts()

0    6443
1    5165
Name: MDR_label, dtype: int64

In [ ]:
MDR_stay_id.to_csv('/MIMIC_IV/2_MDR_label.csv',index=False)

In [ ]:
MDR_stay_id = pd.read_csv('/MIMIC_IV/2_MDR_label.csv')

In [91]:
MDR_stay_id.columns = ['ICUSTAY_ID', 'Acinetobacter spp.', 'Enterobacteriaceae',
       'Enterococcus spp.', 'Pseudomonas aeruginosa', 'Staphylococcus aureus',
       'MDR_label']

In [ ]:
all_ts = pd.read_csv('/MIMIC_IV/Median_B/All_TS_non_49023.csv')

In [84]:
len(all_ts.ICUSTAY_ID.unique())

49023

In [92]:
all_ts = pd.merge(all_ts[['ICUSTAY_ID']],MDR_stay_id,on='ICUSTAY_ID',how='left')
all_ts.shape

(49023, 7)

In [93]:
all_ts.head()

,ICUSTAY_ID,Acinetobacter spp.,Enterobacteriaceae,Enterococcus spp.,Pseudomonas aeruginosa,Staphylococcus aureus,MDR_label
0,30000153,NaN,NaN,NaN,NaN,NaN,NaN
1,30000213,NaN,NaN,NaN,NaN,NaN,NaN
2,30000484,NaN,NaN,NaN,NaN,NaN,NaN
3,30000646,NaN,NaN,NaN,NaN,NaN,NaN
4,30001148,NaN,NaN,NaN,NaN,NaN,NaN


In [94]:
all_ts.MDR_label.value_counts()

0.0    5023
1.0    3961
Name: MDR_label, dtype: int64

In [95]:
all_ts['MDR_checked'] = all_ts['MDR_label'].fillna('no_check')

In [96]:
all_ts.head()

,ICUSTAY_ID,Acinetobacter spp.,Enterobacteriaceae,Enterococcus spp.,Pseudomonas aeruginosa,Staphylococcus aureus,MDR_label,MDR_checked
0,30000153,NaN,NaN,NaN,NaN,NaN,NaN,no_check
1,30000213,NaN,NaN,NaN,NaN,NaN,NaN,no_check
2,30000484,NaN,NaN,NaN,NaN,NaN,NaN,no_check
3,30000646,NaN,NaN,NaN,NaN,NaN,NaN,no_check
4,30001148,NaN,NaN,NaN,NaN,NaN,NaN,no_check


In [101]:
all_ts.MDR_checked.value_counts()

no_check    40039
0.0          5023
1.0          3961
Name: MDR_checked, dtype: int64

In [ ]:
SSC_ICU = pd.read_csv('/MIMIC_IV/1_SSC_ICU_IV.csv')
print(SSC_ICU.shape,SSC_ICU.columns)
len(SSC_ICU.stay_id.unique())

(531643, 20) Index(['microevent_id', 'micro_specimen_id', 'subject_id', 'hadm_id',
       'chartdate', 'charttime', 'spec_itemid', 'spec_type_desc', 'storedate',
       'storetime', 'test_name', 'org_itemid', 'org_name', 'ab_name',
       'interpretation', 'stay_id', 'intime', 'outtime', 'admittime', 'los'],
      dtype='object')


56899

In [108]:
all_ts.shape

(49023, 8)

In [109]:
all_ts = all_ts[all_ts.ICUSTAY_ID.isin(SSC_ICU.stay_id.unique())]

In [110]:
all_ts.shape

(40815, 8)

In [111]:
all_ts.MDR_checked.value_counts()

no_check    31831
0.0          5023
1.0          3961
Name: MDR_checked, dtype: int64

# ICU - EAT

In [ ]:
ABrx = pd.read_csv('/MIMIC_IV/MIMICIV_31_ABrx.csv', dtype={'ndc': str})
ABrx.shape

(992231, 23)

In [61]:
len(ABrx.hadm_id.unique())

245644

In [62]:
ABrx = ABrx.reset_index().rename(columns={'index': 'ab_id'})
ABrx.rename(index=str, columns={ 'starttime':'ab_start',
                                    'stoptime':'ab_end'}, inplace=True)

In [63]:
ABrx.columns

Index(['ab_id', 'subject_id', 'hadm_id', 'pharmacy_id', 'poe_id', 'poe_seq',
       'order_provider_id', 'ab_start', 'ab_end', 'drug_type', 'drug',
       'formulary_drug_cd', 'gsn', 'ndc', 'prod_strength', 'form_rx',
       'dose_val_rx', 'dose_unit_rx', 'form_val_disp', 'form_unit_disp',
       'doses_per_24_hrs', 'route', 'rxcui', 'Antibiotics'],
      dtype='object')

In [68]:
pts.shape

(74829, 30)

In [69]:
pts.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit',
       'intime', 'outtime', 'los', 'gender', 'anchor_age', 'anchor_year',
       'anchor_year_group', 'dod', 'yob', 'min_valid_year', 'admittime',
       'dischtime', 'deathtime', 'admission_type', 'admit_provider_id',
       'admission_location', 'discharge_location', 'insurance', 'language',
       'marital_status', 'race', 'edregtime', 'edouttime',
       'hospital_expire_flag', 'Age'],
      dtype='object')

In [ ]:
ABrxmerged = pd.merge(ABrx, pts, on=['subject_id', 'hadm_id'], how='left')

ABrxmerged['time_match'] = (ABrxmerged['ab_start'] >= ABrxmerged['intime']) & (ABrxmerged['ab_start'] < ABrxmerged['outtime'])

result = ABrxmerged[ABrxmerged['time_match']].copy()

In [71]:
#removing abrx where startdate>enddate
crit = result['ab_start'] <= result['ab_end']
ABrx_ntnull_icu= result[crit]
ABrx_ntnull_icu.shape

(248608, 53)

In [72]:
ABrx_ntnull_icu.Antibiotics.value_counts()

True    248608
Name: Antibiotics, dtype: int64

In [74]:
print(ABrx_ntnull_icu.columns.tolist())

['ab_id', 'subject_id', 'hadm_id', 'pharmacy_id', 'poe_id', 'poe_seq', 'order_provider_id', 'ab_start', 'ab_end', 'drug_type', 'drug', 'formulary_drug_cd', 'gsn', 'ndc', 'prod_strength', 'form_rx', 'dose_val_rx', 'dose_unit_rx', 'form_val_disp', 'form_unit_disp', 'doses_per_24_hrs', 'route', 'rxcui', 'Antibiotics', 'stay_id', 'first_careunit', 'last_careunit', 'intime', 'outtime', 'los', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod', 'yob', 'min_valid_year', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admit_provider_id', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'edregtime', 'edouttime', 'hospital_expire_flag', 'Age', 'time_match']


In [75]:
ABrx_ntnull_icu = ABrx_ntnull_icu[['ab_id', 'subject_id', 'hadm_id', 'pharmacy_id',
                                   'poe_id', 'poe_seq', 'order_provider_id', 
                                   'ab_start', 'ab_end', 'drug_type', 'drug', 
                                   'formulary_drug_cd', 'gsn', 'ndc', 
                                   'prod_strength', 'form_rx', 'dose_val_rx', 
                                   'dose_unit_rx', 'form_val_disp', 
                                   'form_unit_disp', 'doses_per_24_hrs', 
                                   'route', 'rxcui', 'Antibiotics', 
                                   'stay_id', 'first_careunit', 'last_careunit', 
                                   'intime', 'outtime', 'los', 'gender', 
                                   'admittime', 'admission_type', 'admit_provider_id', 
                                   'admission_location', 'discharge_location', 
                                   'insurance', 'language', 'marital_status', 
                                   'race', 'edregtime', 'edouttime', 
                                   'hospital_expire_flag', 'Age', 'time_match']]

In [ ]:
ABrx_ntnull_icu.to_csv('/MIMIC_IV/MIMICIV_31_ABrx_insideicu.csv',index=False)

In [77]:
EAT_icu = ABrx_ntnull_icu.copy()

In [79]:
EAT_icu.time_match.value_counts()

True    248608
Name: time_match, dtype: int64

In [80]:
EAT_icu = EAT_icu.sort_values(['stay_id','ab_start']).drop_duplicates(['stay_id','ab_start','ab_end', 'ndc'],keep='first')

In [81]:
print(
len(EAT_icu), 
len(EAT_icu['stay_id'].unique()),  
len(EAT_icu['ab_id'].unique())
)

246516 52233 246516


In [ ]:
EAT_icu[EAT_icu.stay_id.isin(all_ts.ICUSTAY_ID)].shape

(192445, 45)

In [98]:
EAT_icu.head(2)

,ab_id,subject_id,hadm_id,pharmacy_id,poe_id,poe_seq,order_provider_id,ab_start,ab_end,drug_type,...,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag,Age,time_match
926929,836148,18421337,22413411,3871777,18421337-815,815.0,P14FKD,2136-01-14 21:00:00,2136-01-15 20:00:00,MAIN,...,CHRONIC/LONG TERM ACUTE CARE,Medicare,Other,SINGLE,MULTIPLE RACE/ETHNICITY,2136-01-14 12:18:00,2136-01-14 18:31:00,0.0,92.0,True
926934,836153,18421337,22413411,42000534,18421337-817,817.0,P14FKD,2136-01-14 21:00:00,2136-01-15 16:00:00,MAIN,...,CHRONIC/LONG TERM ACUTE CARE,Medicare,Other,SINGLE,MULTIPLE RACE/ETHNICITY,2136-01-14 12:18:00,2136-01-14 18:31:00,0.0,92.0,True


In [ ]:
EAT_icu.to_csv('/MIMIC_IV/Median_B/1_EAT_insideicu_IV.csv',index=False)

In [ ]:
EAT_icu['ab_start'] = pd.to_datetime(EAT_icu['ab_start'])
EAT_icu['intime'] = pd.to_datetime(EAT_icu['intime'])

EAT_icu['ab_begin'] = (EAT_icu['ab_start'] - EAT_icu['intime']).dt.total_seconds() / 3600  # 转换为小时

EAT_icu_filtered = EAT_icu[EAT_icu['ab_begin'] <= 48]

In [114]:
EAT_icu_filtered.shape

(148826, 46)

In [115]:
ab_count_filtered = EAT_icu_filtered.groupby('stay_id')['ab_id'].nunique().reset_index()
ab_count_filtered.columns = ['ICUSTAY_ID', 'ab_id_filtered_count']

In [116]:
ab_count = EAT_icu.groupby('stay_id')['ab_id'].nunique().reset_index()
ab_count.columns = ['ICUSTAY_ID', 'ab_id_count']

In [117]:
fina_ab_count = pd.merge(ab_count,ab_count_filtered,on='ICUSTAY_ID',how='left')

In [118]:
fina_ab_count['ATE'] = fina_ab_count.apply(lambda row: 1 if pd.notna(row['ab_id_count']) or pd.notna(row['ab_id_filtered_count']) else 0, axis=1)
fina_ab_count['ATE_filtered'] = fina_ab_count.apply(lambda row: 1 if pd.notna(row['ab_id_filtered_count']) else 0, axis=1)

In [ ]:
fina_ab_count.to_csv('MIMIC_IV/Median_B/All_AET_count.csv',index=False)

In [127]:
all_ts = pd.merge(all_ts,fina_ab_count,on='ICUSTAY_ID',how='left')
all_ts.shape

(40815, 12)

In [129]:
all_ts = all_ts.fillna(0)

In [ ]:
all_ts.to_csv('/MIMIC_IV/Median_B/Label_MDR_ATE.csv',index=False)